In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 1: Setup with Model Selection
# ══════════════════════════════════════════════════════════════════════════════

!pip install anthropic -q

import anthropic
import json
import re

claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# Use a DIFFERENT model from the one used to generate responses (Sonnet 4)
JUDGE_MODEL = "claude-opus-4-6"  # Opus 4.6

print(f"✅ Claude API initialized")
print(f"📊 Judge model: {JUDGE_MODEL}")

✅ Claude API initialized
📊 Judge model: claude-opus-4-6


In [18]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 2: Robust LLM-as-a-Judge Evaluation Function
# ══════════════════════════════════════════════════════════════════════════════

def evaluate_responses(test_name: str, query: str, rag_response: str, general_response: str) -> dict:
    """
    Use a different Claude model as judge to evaluate RAG vs General LLM responses.
    Includes robust JSON parsing with fallbacks and detailed error handling.
    """

    prompt = f"""You are an expert evaluator assessing two AI responses about Singapore birds.

TEST CONTEXT:
Test Name: {test_name}
User Query: {query}

RESPONSE A (RAG System - uses retrieved Singapore-specific data):
{rag_response}

RESPONSE B (General LLM - no retrieval, parametric knowledge only):
{general_response}

EVALUATION CRITERIA:
Score each response from 1-5 on these criteria:

1. SINGAPORE_SPECIFICITY
   - Does it mention Singapore locations, local conservation status, or local data sources?
   - 5 = Multiple specific Singapore references (parks, survey data, local status)
   - 1 = No Singapore-specific information, only generic/global information

2. FACTUAL_ACCURACY
   - Are the stated facts correct and verifiable?
   - 5 = All facts accurate and verifiable
   - 1 = Contains clear factual errors

3. GROUNDEDNESS
   - Is the information traceable to a specific source?
   - 5 = Explicitly cites sources (e.g., "NParks report", "Garden Birdwatch survey")
   - 1 = No sources mentioned, purely parametric knowledge

4. NO_HALLUCINATION
   - Does it avoid plausible but unverifiable claims?
   - 5 = All claims are verifiable or appropriately hedged
   - 1 = Contains confident claims that cannot be verified

5. COMPLETENESS
   - Does it cover all information categories relevant to the query?
   - 5 = Comprehensive coverage of all relevant aspects
   - 1 = Missing major relevant information

IMPORTANT: Respond ONLY with valid JSON. No markdown backticks, no explanations before or after.

{{
    "response_a_scores": {{
        "singapore_specificity": <int 1-5>,
        "factual_accuracy": <int 1-5>,
        "groundedness": <int 1-5>,
        "no_hallucination": <int 1-5>,
        "completeness": <int 1-5>
    }},
    "response_b_scores": {{
        "singapore_specificity": <int 1-5>,
        "factual_accuracy": <int 1-5>,
        "groundedness": <int 1-5>,
        "no_hallucination": <int 1-5>,
        "completeness": <int 1-5>
    }},
    "justifications": {{
        "singapore_specificity": "<one sentence comparison>",
        "factual_accuracy": "<one sentence comparison>",
        "groundedness": "<one sentence comparison>",
        "no_hallucination": "<one sentence comparison>",
        "completeness": "<one sentence comparison>"
    }},
    "overall_winner": "<Response A or Response B or Tie>",
    "summary": "<two sentence summary>"
}}"""

    # Step 1: Make API call with error handling
    try:
        response = claude.messages.create(
            model=JUDGE_MODEL,
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )
        raw_text = response.content[0].text
        print(f"   ✓ API call successful ({len(raw_text)} chars)")

    except Exception as e:
        print(f"   ❌ API Error: {type(e).__name__}: {e}")
        return {
            "test_name": test_name,
            "query": query,
            "parse_success": False,
            "error_type": "api_error",
            "raw_response": str(e)
        }

    # Step 2: Clean up response - remove markdown code blocks and extra text
    cleaned_text = raw_text.strip()

    # Remove markdown code blocks
    cleaned_text = re.sub(r'^```json\s*\n?', '', cleaned_text)
    cleaned_text = re.sub(r'^```\s*\n?', '', cleaned_text)
    cleaned_text = re.sub(r'\n?```\s*$', '', cleaned_text)

    # Try to extract JSON if there's text before or after
    json_match = re.search(r'\{[\s\S]*\}', cleaned_text)
    if json_match:
        cleaned_text = json_match.group(0)

    cleaned_text = cleaned_text.strip()

    # Step 3: Parse JSON with error handling
    try:
        result = json.loads(cleaned_text)
        result["test_name"] = test_name
        result["query"] = query
        result["parse_success"] = True
        return result

    except json.JSONDecodeError as e:
        print(f"   ⚠️ JSON parse error: {e}")
        print(f"   Raw preview: {raw_text[:200]}...")
        return {
            "test_name": test_name,
            "query": query,
            "parse_success": False,
            "error_type": "json_parse_error",
            "raw_response": raw_text
        }


print("✅ Evaluation function defined")
print(f"   Judge model: {JUDGE_MODEL}")

✅ Evaluation function defined
   Judge model: claude-opus-4-6


In [19]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 3: Define Test Cases with Responses
# ══════════════════════════════════════════════════════════════════════════════

# Paste your actual responses here
test_cases = [
    {
        "test_name": "Test 1: Species Explanation (Tiger Shrike)",
        "query": "A bird sound recognition system has identified: Tiger Shrike (87% confidence). Provide educational information.",
        "rag_response": """What an exciting identification! The Tiger Shrike is a fascinating passage migrant that you're lucky to encounter in Singapore.

**Physical Description:** Adult Tiger Shrikes are quite distinctive with their bold black face mask, grey crown and mantle, and beautiful chestnut-brown upperparts heavily decorated with black barring. Their underparts are clean white with a subtle yellowish tinge. If you spot a more uniformly brown bird with black barring above and scaling below, that's likely an immature Tiger Shrike - they're actually seen more frequently than adults and lack the striking face mask.

**Where to Find Them:** Unlike their Brown Shrike cousins who prefer open areas, Tiger Shrikes favor heavily wooded parks. The Garden Birdwatch recorded them at 23 sites, with Mount Faber and Coney Island Park being prime locations. Listen for their harsh chattering call - it often reveals their presence before you spot them!

**Migration & Conservation:** These birds are passage migrants, peaking between late October and early November as they journey through Singapore en route to Indonesia. They're classified as Near Threatened, making each sighting particularly special for conservation efforts.""",

        "general_response": """What an exciting identification! The Tiger Shrike is a striking bird that lives up to its fierce name. This medium-sized shrike displays beautiful russet-brown plumage with distinctive black barring across its back and wings, giving it that "tiger-striped" appearance. Males sport a bold black mask through their eyes, while females show more subdued coloring.

You'll find Tiger Shrikes across East Asia, from eastern Russia and China down through Korea and into Southeast Asia. They're primarily migratory birds, breeding in the northern parts of their range during summer and heading south to warmer regions like Thailand, Malaysia, and southern China for winter.

Currently listed as "Least Concern" by conservation organizations, Tiger Shrikes are doing relatively well, though habitat loss remains a concern in some areas.

Here's a fascinating fact: like all shrikes, they're known as "butcher birds" for their habit of impaling prey – insects, small reptiles, and even mice – on thorns or barbed wire to create natural "pantries"! This behavior helps them tear apart their food and store it for later meals.

Happy birding, and great spot!"""
    },
    {
        "test_name": "Test 2: Species-Specific Habitat (Collared Kingfisher)",
        "query": "Where can I spot the Collared Kingfisher?",
        "rag_response": """Based on the NParks Garden Birdwatch report, you can spot Collared Kingfishers in a wonderfully diverse range of locations across Singapore!

**Best spots include:**
- **Coastal areas** like East Coast Park and Sungei Buloh Wetland Reserve (where the highest counts are recorded)
- **Urban areas** throughout the island - this adaptable species was recorded at 60 survey sites
- **Wooded habitats** with tall trees that serve as hunting perches
- **Building rooftops** where they perch on antennae as substitutes for natural perches

This resident species has brilliantly adapted from its traditional coastal habitat to thrive in Singapore's urban greenery. You'll often see them using tall trees or even building antennae as vantage points for hunting.

**Timing tip:** The report suggests they're more numerous during April (breeding season), though most sightings occur during November surveys when migratory individuals join the resident population.

The Collared Kingfisher's remarkable urban adaptation means you have excellent chances of spotting this blue-and-white beauty whether you're exploring coastal parks or simply looking up at city rooftops!""",

        "general_response": """The Collared Kingfisher is wonderfully widespread across tropical Asia and the western Pacific! You'll find these striking blue-and-white birds from India and Southeast Asia through Indonesia, the Philippines, and out to many Pacific islands including northern Australia.

Unlike their fish-loving cousins, Collared Kingfishers are quite adaptable about habitat. Look for them in:

- **Coastal areas** - mangroves, beaches, and tidal flats
- **Urban parks and gardens** with large trees
- **Forest edges** and clearings
- **Rice paddies** and agricultural areas
- **Wooded areas** near water sources

They're often spotted perched prominently on power lines, dead branches, or posts, scanning for insects, small crabs, and lizards. Listen for their loud, rattling call - it's quite distinctive!

The best viewing times are early morning and late afternoon when they're most active. In many areas, they've adapted well to human presence, so don't be surprised to see them in busy parks or even suburban neighborhoods.

Each region has slight variations in their appearance, which makes spotting them across their range particularly rewarding for birders. They're generally year-round residents, though some northern populations may migrate seasonally."""
    },
    {
        "test_name": "Test 3: Cross-Species Categorical (Migratory Birds)",
        "query": "Which birds in Singapore are migratory?",
        "rag_response": """Based on the NParks Garden Birdwatch report, **seven migratory bird species** are featured in Singapore:

**Bee-eaters & Flycatchers:**
- Blue-tailed Bee-eater - Common passage migrant, peak in November
- Asian Paradise Flycatcher - Passage migrant during northern winter
- Asian Brown Flycatcher - Most common migratory flycatcher, adaptable to urban areas

**Shrikes:**
- Tiger Shrike - Common passage migrant, prefers heavily wooded areas
- Brown Shrike - Common migrant favouring open grasslands and lawns

**Warblers & Starlings:**
- Arctic Warbler - Described as "arguably the most abundant migratory landbird" wintering in Singapore
- Daurian Starling - Common passage migrant and winter visitor, often in mixed flocks

These species primarily breed in northern regions like eastern Russia, Siberia, and East Asia, then migrate to Southeast Asia during the northern winter. **Peak migration occurs between late October and early November**, with some individuals overwintering in Singapore before returning to breeding grounds by April.

The report notes that Singapore serves as both a passage route for birds heading further south to Indonesia and a winter refuge for those staying the season.""",

        "general_response": """Singapore hosts many fascinating migratory birds, particularly during the October-March winter months when species escape colder northern regions!

**Common migrants include:**
- **Raptors**: Chinese Goshawk, Japanese Sparrowhawk, and Grey-faced Buzzard pass through during spectacular migration periods
- **Shorebirds**: Various sandpipers, plovers, and turnstones stop at mudflats and coastlines
- **Forest birds**: Asian Paradise Flycatcher, Tiger Shrike, and several warbler species
- **Waterbirds**: Northern Pintail, Garganey ducks, and Chinese Pond Heron

The **Sungei Buloh Wetland Reserve** is Singapore's premier migration hotspot, where thousands of shorebirds refuel during their epic journeys between Arctic breeding grounds and Australian wintering areas.

Some exciting seasonal visitors include the stunning **Blue-winged Pitta** and various **leaf warblers** that brighten our parks and nature reserves.

Singapore's position along the East Asian-Australasian Flyway makes it a crucial stopover point. Peak migration viewing is typically September-November and February-April.

For the best experience, visit early morning with binoculars and check eBird for recent sightings. The diversity changes seasonally, so there's always something new to discover!

Happy birding! 🦅"""
    },
    {
        "test_name": "Test 4: Location-Based Query (Ang Mo Kio)",
        "query": "I live in Ang Mo Kio.",
        "rag_response": """Hello! Since you live in Ang Mo Kio, you're in a great location for birdwatching! Based on the NParks Garden Birdwatch report, here are some birds you're likely to encounter in your area:

**Most Common:**
- **Yellow-vented Bulbul** - This abundant resident was recorded at all survey sites! Look for its distinctive white head with black markings and yellow vent.
- **Javan Myna** - Singapore's most recognisable bird with black plumage, yellow eyes, bill and legs, plus white wing patches. This introduced species is everywhere.

**Also Likely:**
- **House Crow** - The large grey and black crow that's widespread throughout Singapore
- **Greater Racket-tailed Drongo** - If you're near any wooded areas or parks, watch for this distinctive bird with its modified tail feathers

The report shows that adaptable species like the Yellow-vented Bulbul thrive across Singapore's varied habitats, from nature parks to urban areas like the Central Business District. Since Ang Mo Kio has several parks and green spaces, you should have good opportunities to spot these common residents along with the introduced species that have made Singapore home.

Happy birdwatching in your neighbourhood!""",

        "general_response": """Hello fellow bird enthusiast! Ang Mo Kio is a wonderful area for birdwatching in Singapore. You're in for some great birding opportunities right in your neighborhood!

**Common species you can spot:**
- **Javan Mynas** and **Common Mynas** - abundant around HDB blocks
- **Pink-necked Green Pigeons** - look for them in fruiting trees
- **Yellow-vented Bulbuls** - very common, with their distinctive calls
- **Oriental Magpie-Robins** - beautiful songbirds often seen in parks
- **Zebra Doves** - ground-feeding doves with soft cooing calls

**Great birding spots near you:**
- **Ang Mo Kio Town Garden West** - excellent for various species
- **Bishan-Ang Mo Kio Park** - fantastic diversity, especially near the river
- **Upper Seletar Reservoir** area - for waterbirds and forest species

Early mornings (6-9 AM) are prime time for bird activity. Bring binoculars if you have them, and listen for calls - you'll often hear birds before seeing them!

The area's mix of urban greenery and park connectors creates excellent bird corridors. Keep an eye out for seasonal migrants too, especially during September-November and March-May.

Happy birding in your neighborhood!"""
    },
    {
        "test_name": "Test 5: Subjective Query (Cutest Bird)",
        "query": "What is the cutest bird?",
        "rag_response": """I appreciate this delightful question! However, the NParks Garden Birdwatch report I have access to doesn't include subjective assessments of which birds are "cutest" - it focuses on scientific data like distribution, abundance, and conservation status.

That said, from the descriptions provided, there are some particularly charming candidates! The **Scarlet-backed Flowerpecker** is described as "one of Singapore's smallest and most attractive urban birds" with males having a striking scarlet crown and glossy blue wings. At the other end of the size spectrum, the **Red Junglefowl** could win hearts as the "ancestor of the domestic chicken" - who doesn't find baby chicks adorable?

The **Pink-necked Green Pigeon** also sounds lovely, with males having an attractive greyish face transitioning to a pinkish neck and orange breast patch.

Since "cuteness" is quite subjective and depends on personal preference - whether you're drawn to tiny, colorful birds or find larger, more familiar species endearing - I'd encourage you to observe these species yourself in Singapore's parks and gardens to decide which tugs at your heartstrings most!

The report's strength lies in its valuable conservation and behavioral insights rather than aesthetic judgments.""",

        "general_response": """What a delightful question! While "cuteness" is wonderfully subjective, several birds consistently melt hearts worldwide.

**Puffins** are strong contenders with their colorful beaks, tuxedo plumage, and clumsy-adorable waddle. Their expressive orange eyes and perpetually curious expression make them irresistible.

**Northern Saw-whet Owls** are impossibly tiny (sparrow-sized!) with enormous golden eyes and fluffy, round bodies that look like living stuffed animals.

**Japanese Long-tailed Tits** resemble floating cotton balls with their incredibly fluffy white plumage and tiny black button eyes.

**Bee Hummingbirds** (world's smallest bird) are so petite they're often mistaken for insects, hovering like jeweled sprites.

**Baby chickens** deserve mention too – those downy yellow fluff balls with their tiny cheeps are universally adored.

Honestly, "cuteness" depends on what appeals to you: Is it big eyes? Fluffy feathers? Tiny size? Clumsy behavior? Every birder has their favorite, and that's part of the joy of birdwatching – discovering which species capture your heart.

What type of "cute" draws you to birds? I'd love to suggest some species that match your preferences!"""
    }
]

print(f"✅ {len(test_cases)} test cases defined")

✅ 5 test cases defined


In [20]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 4: Run Evaluation
# ══════════════════════════════════════════════════════════════════════════════

print(f"Running LLM-as-a-Judge evaluation using {JUDGE_MODEL}...\n")
print("=" * 80)

all_results = []

for test in test_cases:
    print(f"\n📊 Evaluating: {test['test_name']}")

    result = evaluate_responses(
        test_name=test["test_name"],
        query=test["query"],
        rag_response=test["rag_response"],
        general_response=test["general_response"]
    )

    all_results.append(result)

    if result.get("parse_success", False):
        rag_total = sum(result["response_a_scores"].values())
        llm_total = sum(result["response_b_scores"].values())
        print(f"   RAG: {rag_total}/25 | General LLM: {llm_total}/25 | Winner: {result['overall_winner']}")
    else:
        print(f"   ⚠️ Parse failed - check raw response")

print("\n" + "=" * 80)
print("✅ Evaluation complete!")

Running LLM-as-a-Judge evaluation using claude-opus-4-6...


📊 Evaluating: Test 1: Species Explanation (Tiger Shrike)
   ✓ API call successful (2454 chars)
   RAG: 22/25 | General LLM: 11/25 | Winner: Response A

📊 Evaluating: Test 2: Species-Specific Habitat (Collared Kingfisher)
   ✓ API call successful (2229 chars)
   RAG: 22/25 | General LLM: 12/25 | Winner: Response A

📊 Evaluating: Test 3: Cross-Species Categorical (Migratory Birds)
   ✓ API call successful (2115 chars)
   RAG: 24/25 | General LLM: 14/25 | Winner: Response A

📊 Evaluating: Test 4: Location-Based Query (Ang Mo Kio)
   ✓ API call successful (2301 chars)
   RAG: 21/25 | General LLM: 18/25 | Winner: Response A

📊 Evaluating: Test 5: Subjective Query (Cutest Bird)
   ✓ API call successful (2285 chars)
   RAG: 22/25 | General LLM: 16/25 | Winner: Response A

✅ Evaluation complete!


In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# DEBUG: Check raw responses
# ══════════════════════════════════════════════════════════════════════════════

for result in all_results:
    if not result.get("parse_success", False):
        print(f"\n{'='*60}")
        print(f"Test: {result['test_name']}")
        print(f"{'='*60}")
        print("RAW RESPONSE:")
        print(result.get("raw_response", "No raw response saved")[:500])
        print("...")


Test: Test 1: Species Explanation (Tiger Shrike)
RAW RESPONSE:
No raw response saved
...

Test: Test 2: Species-Specific Habitat (Collared Kingfisher)
RAW RESPONSE:
No raw response saved
...


In [21]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 5: Generate Summary Table
# ══════════════════════════════════════════════════════════════════════════════

import pandas as pd

# Build summary dataframe
summary_data = []

for result in all_results:
    if "error" not in result:
        rag_scores = result["response_a_scores"]
        llm_scores = result["response_b_scores"]

        summary_data.append({
            "Test": result["test_name"].replace("Test ", "").split(":")[0],
            "RAG_Singapore": rag_scores["singapore_specificity"],
            "LLM_Singapore": llm_scores["singapore_specificity"],
            "RAG_Accuracy": rag_scores["factual_accuracy"],
            "LLM_Accuracy": llm_scores["factual_accuracy"],
            "RAG_Grounded": rag_scores["groundedness"],
            "LLM_Grounded": llm_scores["groundedness"],
            "RAG_NoHalluc": rag_scores["no_hallucination"],
            "LLM_NoHalluc": llm_scores["no_hallucination"],
            "RAG_Complete": rag_scores["completeness"],
            "LLM_Complete": llm_scores["completeness"],
            "RAG_Total": sum(rag_scores.values()),
            "LLM_Total": sum(llm_scores.values()),
            "Winner": result["overall_winner"]
        })

df = pd.DataFrame(summary_data)

print("=" * 80)
print("SUMMARY TABLE: LLM-as-a-Judge Evaluation Results")
print("=" * 80)
print(df.to_string(index=False))

# Calculate averages
print("\n" + "-" * 80)
print("AGGREGATE SCORES")
print("-" * 80)
print(f"RAG System Average Total:     {df['RAG_Total'].mean():.2f} / 25")
print(f"General LLM Average Total:    {df['LLM_Total'].mean():.2f} / 25")
print(f"\nRAG Wins: {(df['Winner'] == 'Response A').sum()}")
print(f"LLM Wins: {(df['Winner'] == 'Response B').sum()}")
print(f"Ties:     {(df['Winner'] == 'Tie').sum()}")

SUMMARY TABLE: LLM-as-a-Judge Evaluation Results
Test  RAG_Singapore  LLM_Singapore  RAG_Accuracy  LLM_Accuracy  RAG_Grounded  LLM_Grounded  RAG_NoHalluc  LLM_NoHalluc  RAG_Complete  LLM_Complete  RAG_Total  LLM_Total     Winner
   1              5              1             4             3             4             1             4             3             5             3         22         11 Response A
   2              5              1             4             4             5             1             3             3             5             3         22         12 Response A
   3              5              3             5             3             5             1             5             3             4             4         24         14 Response A
   4              5              5             4             4             5             1             4             3             3             5         21         18 Response A
   5              5              1             4   

In [22]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 6: Print Detailed Justifications
# ══════════════════════════════════════════════════════════════════════════════

print("=" * 80)
print("DETAILED JUSTIFICATIONS")
print("=" * 80)

for result in all_results:
    if result.get("parse_success", False):
        print(f"\n{'─' * 80}")
        print(f"📌 {result['test_name']}")
        print(f"{'─' * 80}")

        for criterion, justification in result["justifications"].items():
            rag_score = result["response_a_scores"][criterion]
            llm_score = result["response_b_scores"][criterion]
            print(f"\n  {criterion.replace('_', ' ').title()}")
            print(f"    RAG: {rag_score}/5 | LLM: {llm_score}/5")
            print(f"    → {justification}")

        print(f"\n  🏆 Winner: {result['overall_winner']}")
        print(f"  📝 Summary: {result['summary']}")

DETAILED JUSTIFICATIONS

────────────────────────────────────────────────────────────────────────────────
📌 Test 1: Species Explanation (Tiger Shrike)
────────────────────────────────────────────────────────────────────────────────

  Singapore Specificity
    RAG: 5/5 | LLM: 1/5
    → Response A mentions specific Singapore locations (Mount Faber, Coney Island Park), local survey data (Garden Birdwatch, 23 sites), and local migration patterns, while Response B provides only generic range information with no Singapore-specific content.

  Factual Accuracy
    RAG: 4/5 | LLM: 3/5
    → Response A's details about plumage, passage migrant status, and migration timing align well with known Singapore ornithological data, though the Near Threatened classification may be debatable; Response B incorrectly states the IUCN status as Least Concern when the Tiger Shrike is actually listed as Least Concern globally but its local Singapore status differs, and its range description is somewhat oversim

In [23]:
# ══════════════════════════════════════════════════════════════════════════════
# CELL 7: Export Results for Appendix
# ══════════════════════════════════════════════════════════════════════════════

from datetime import datetime

# Filter successful results
successful_results = [r for r in all_results if r.get("parse_success", False)]

if len(successful_results) == 0:
    print("❌ No successful evaluations to export.")
else:
    # ── Export 1: JSON (machine-readable) ─────────────────────────────────────
    with open("appendix_llm_judge_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print("✅ Saved: appendix_llm_judge_results.json")

    # ── Export 2: Formatted Text Report (for appendix) ────────────────────────
    report_filename = "appendix_llm_judge_report.txt"

    with open(report_filename, "w") as f:
        # Header
        f.write("=" * 80 + "\n")
        f.write("APPENDIX: LLM-AS-A-JUDGE EVALUATION REPORT\n")
        f.write("=" * 80 + "\n\n")

        f.write("EVALUATION METADATA\n")
        f.write("-" * 40 + "\n")
        f.write(f"Evaluation Date:      {datetime.now().strftime('%Y-%m-%d %H:%M')}\n")
        f.write(f"Judge Model:          {JUDGE_MODEL}\n")
        f.write(f"Response Model:       claude-sonnet-4-20250514\n")
        f.write(f"Tests Evaluated:      {len(successful_results)}\n\n")

        # Evaluation Criteria Definitions
        f.write("EVALUATION CRITERIA\n")
        f.write("-" * 40 + "\n")
        f.write("1. Singapore-Specificity: Mentions Singapore locations, local status, or NParks data\n")
        f.write("2. Factual Accuracy:      Facts are correct and verifiable\n")
        f.write("3. Groundedness:          Information traceable to specific sources\n")
        f.write("4. No Hallucination:      Absence of plausible but unverifiable claims\n")
        f.write("5. Completeness:          Covers all requested information categories\n")
        f.write("\nScoring: 1 (Poor) to 5 (Excellent)\n\n")

        # Summary Table
        f.write("=" * 80 + "\n")
        f.write("SUMMARY OF RESULTS\n")
        f.write("=" * 80 + "\n\n")

        f.write(f"{'Test':<45} {'RAG':>8} {'LLM':>8} {'Winner':>12}\n")
        f.write("-" * 75 + "\n")

        rag_totals = []
        llm_totals = []

        for result in successful_results:
            rag_total = sum(result["response_a_scores"].values())
            llm_total = sum(result["response_b_scores"].values())
            rag_totals.append(rag_total)
            llm_totals.append(llm_total)

            test_short = result["test_name"][:44]
            f.write(f"{test_short:<45} {rag_total:>6}/25 {llm_total:>6}/25 {result['overall_winner']:>12}\n")

        f.write("-" * 75 + "\n")
        f.write(f"{'AVERAGE':<45} {sum(rag_totals)/len(rag_totals):>7.1f} {sum(llm_totals)/len(llm_totals):>7.1f}\n\n")

        # Per-Criterion Summary
        f.write("PER-CRITERION AVERAGES\n")
        f.write("-" * 40 + "\n")

        criteria = ["singapore_specificity", "factual_accuracy", "groundedness", "no_hallucination", "completeness"]
        criteria_labels = ["Singapore-Specificity", "Factual Accuracy", "Groundedness", "No Hallucination", "Completeness"]

        for criterion, label in zip(criteria, criteria_labels):
            rag_avg = sum(r["response_a_scores"][criterion] for r in successful_results) / len(successful_results)
            llm_avg = sum(r["response_b_scores"][criterion] for r in successful_results) / len(successful_results)
            diff = rag_avg - llm_avg
            winner = "RAG" if diff > 0.1 else ("LLM" if diff < -0.1 else "Tie")
            f.write(f"{label:<25} RAG: {rag_avg:.2f}  LLM: {llm_avg:.2f}  ({winner})\n")

        f.write("\n")

        # Detailed Results per Test
        f.write("=" * 80 + "\n")
        f.write("DETAILED RESULTS BY TEST\n")
        f.write("=" * 80 + "\n")

        for i, result in enumerate(successful_results, 1):
            f.write(f"\n{'─' * 80}\n")
            f.write(f"TEST {i}: {result['test_name']}\n")
            f.write(f"{'─' * 80}\n\n")

            f.write(f"Query: {result['query']}\n\n")

            # Scores table
            f.write("SCORES:\n")
            f.write(f"{'Criterion':<25} {'RAG':>8} {'LLM':>8}\n")
            f.write("-" * 45 + "\n")

            for criterion, label in zip(criteria, criteria_labels):
                rag_score = result["response_a_scores"][criterion]
                llm_score = result["response_b_scores"][criterion]
                f.write(f"{label:<25} {rag_score:>8} {llm_score:>8}\n")

            rag_total = sum(result["response_a_scores"].values())
            llm_total = sum(result["response_b_scores"].values())
            f.write("-" * 45 + "\n")
            f.write(f"{'TOTAL':<25} {rag_total:>6}/25 {llm_total:>6}/25\n\n")

            f.write(f"Winner: {result['overall_winner']}\n\n")

            # Justifications
            f.write("JUSTIFICATIONS:\n")
            for criterion, label in zip(criteria, criteria_labels):
                justification = result["justifications"].get(criterion, "N/A")
                f.write(f"  {label}:\n")
                f.write(f"    {justification}\n\n")

            f.write(f"Summary: {result['summary']}\n")

        # Response Transcripts
        f.write("\n" + "=" * 80 + "\n")
        f.write("RESPONSE TRANSCRIPTS\n")
        f.write("=" * 80 + "\n")

        for i, test in enumerate(test_cases, 1):
            # Find matching result
            matching_result = next((r for r in successful_results if r["test_name"] == test["test_name"]), None)
            if matching_result:
                f.write(f"\n{'─' * 80}\n")
                f.write(f"TEST {i}: {test['test_name']}\n")
                f.write(f"Query: {test['query']}\n")
                f.write(f"{'─' * 80}\n\n")

                f.write("RESPONSE A (RAG System):\n")
                f.write("-" * 40 + "\n")
                f.write(test["rag_response"])
                f.write("\n\n")

                f.write("RESPONSE B (General LLM):\n")
                f.write("-" * 40 + "\n")
                f.write(test["general_response"])
                f.write("\n")

    print(f"✅ Saved: {report_filename}")

    # ── Export 3: CSV (for tables) ────────────────────────────────────────────
    csv_filename = "appendix_llm_judge_scores.csv"

    csv_data = []
    for result in successful_results:
        row = {
            "Test": result["test_name"],
            "Query": result["query"],
        }
        for criterion in criteria:
            row[f"RAG_{criterion}"] = result["response_a_scores"][criterion]
            row[f"LLM_{criterion}"] = result["response_b_scores"][criterion]
        row["RAG_Total"] = sum(result["response_a_scores"].values())
        row["LLM_Total"] = sum(result["response_b_scores"].values())
        row["Winner"] = result["overall_winner"]
        row["Summary"] = result["summary"]
        csv_data.append(row)

    csv_df = pd.DataFrame(csv_data)
    csv_df.to_csv(csv_filename, index=False)
    print(f"✅ Saved: {csv_filename}")

    # ── Print file locations ──────────────────────────────────────────────────
    print("\n" + "=" * 60)
    print("EXPORTED FILES FOR APPENDIX")
    print("=" * 60)
    print(f"1. {report_filename:<40} - Formatted text report")
    print(f"2. appendix_llm_judge_results.json{'':<9} - Raw JSON data")
    print(f"3. {csv_filename:<40} - CSV for tables")
    print("\nDownload these files and include in your thesis appendix.")

✅ Saved: appendix_llm_judge_results.json
✅ Saved: appendix_llm_judge_report.txt
✅ Saved: appendix_llm_judge_scores.csv

EXPORTED FILES FOR APPENDIX
1. appendix_llm_judge_report.txt            - Formatted text report
2. appendix_llm_judge_results.json          - Raw JSON data
3. appendix_llm_judge_scores.csv            - CSV for tables

Download these files and include in your thesis appendix.


In [24]:
# Download all files
from google.colab import files

files.download("appendix_llm_judge_report.txt")
files.download("appendix_llm_judge_results.json")
files.download("appendix_llm_judge_scores.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>